### Tower Coverage Optimization

The tower coverage optimization problem involves determining the optimal locations to build up to $T$ towers within an $H \times W$ city grid. The objective is to maximize the total population covered. A specific region is considered covered if a tower is built directly within it or in an adjacent neighboring region.

**Parameters:**
* $H$: height of the city grid
* $W$: width of the city grid
* $T$: maximum number of towers that can be built
* $p_{ij}$: population in region $(i,j)$

#### Mathematical Model

**Decision Variables:**
The binary variable $x_{ij}$ equals $1$ if a tower is built in region $(i,j)$, and $0$ otherwise:
$$x_{ij} \in \{0, 1\} \quad \text{for } i=1,\dots,H \text{ and } j=1,\dots,W$$

The binary variable $y_{ij}$ equals $1$ if region $(i,j)$ is covered by at least one tower, and $0$ otherwise:
$$y_{ij} \in \{0, 1\} \quad \text{for } i=1,\dots,H \text{ and } j=1,\dots,W$$

**Objective Function:**
Maximize the total population covered by the network of towers:
$$\text{maximize } z = \sum_{i=1}^{H}\sum_{j=1}^{W}p_{ij}y_{ij}$$

**Constraints:**
1. Coverage Constraint: A region $(i,j)$ can only be counted as covered ($y_{ij}=1$) if a tower is built directly in region $(i,j)$ or in at least one of its neighboring regions $(k,l)$:
$$x_{ij}+\sum_{(k,l)\in\text{neighbors}(i,j)}x_{kl} \ge y_{ij} \quad i=1, 2, \dots, H; \ j=1, 2, \dots, W$$

2. Tower Limit Constraint: The total number of towers built across the entire city grid cannot exceed the maximum allowance $T$:
$$\sum_{i=1}^{H}\sum_{j=1}^{W}x_{ij} \le T$$

In [5]:
# load libraries
import numpy as np
import scipy.sparse as sp

import cplex as cp

In [6]:
def mixed_integer_linear_programming(direction, A, senses, b, c, l, u, types):
    # create an empty optimization problem
    prob = cp.Cplex()

    # add decision variables to the problem including their coefficients in objective and ranges
    prob.variables.add(obj = c.tolist(), lb = l.tolist(), ub = u.tolist(), types = types.tolist())

    # define problem type
    if direction == "maximize":
        prob.objective.set_sense(prob.objective.sense.maximize)
    else:
        prob.objective.set_sense(prob.objective.sense.minimize)

    # add constraints to the problem including their directions and right-hand side values
    prob.linear_constraints.add(senses = senses.tolist(), rhs = b.tolist())

    # add coefficients for each constraint
    row_indices, col_indices = A.nonzero()
    prob.linear_constraints.set_coefficients(zip(row_indices.tolist(), col_indices.tolist(), A.data.tolist()))

    print(prob.write_as_string())
    # solve the problem
    prob.solve()
    
    # check the solution status
    print(prob.solution.get_status())
    print(prob.solution.status[prob.solution.get_status()])

    # get the solution
    x_star = prob.solution.get_values()
    obj_star = prob.solution.get_objective_value()

    return(x_star, obj_star)


In [7]:
def cell_tower_coverage_problem(populations_file, T):
    # your implementation starts below
    populations=np.loadtxt(populations_file)
    H=populations.shape[0]
    W=int(populations.size/H)
    flat=np.array(populations.flatten())
    c=np.concatenate((np.repeat(0, H*W),flat))
    types=np.repeat("B", 2*H*W)
    l=np.repeat(0, 2*H*W)
    u=np.repeat(1, 2*H*W)
    senses=np.concatenate((np.repeat("G", H*W), ["L"]))
    b=np.concatenate((np.repeat(0, H*W), [T]))
    
    A=np.zeros(shape=(W*H+1,W*H*2))
    for i in range(W*(H-1)):
        A[i,W+i]=1
        A[i+W,i]=1
    for i in range(W*H):
        A[i,i]=1
        A[i,W*H+i]=-1
        A[W*H,i]=1
    for i in range(W*H-1):
        for j in range(H):
            if W*j==i+1:
                A[i,i+1]=0
                A[i+1,i]=0
                break
            else:
                A[i,i+1]=1
                A[i+1,i]=1
    A=sp.csr_matrix(A)
    x_star, obj_star = mixed_integer_linear_programming("maximize", A, senses, b, c, l, u, types)
    X_star=np.array(x_star[:H*W]).reshape(H,W)
    # your implementation ends above
    return(X_star)

In [8]:
X_star = cell_tower_coverage_problem("populations.txt", 3)
print(X_star)

Default variable names x1, x2 ... being created.
Default row names c1, c2 ... being created.


\ENCODING=ISO-8859-1
\Problem name: 

Maximize
 obj1: 732 x25 + 539 x26 + 949 x27 + 508 x28 + 806 x29 + 881 x30 + 646 x31
       + 757 x32 + 257 x33 + 630 x34 + 994 x35 + 547 x36 + 105 x37 + 859 x38
       + 876 x39 + 589 x40 + 615 x41 + 345 x42 + 136 x43 + 370 x44 + 433 x45
       + 419 x46 + 631 x47 + 485 x48
Subject To
 c1:  x1 + x2 + x7 - x25 >= 0
 c2:  x1 + x2 + x3 + x8 - x26 >= 0
 c3:  x2 + x3 + x4 + x9 - x27 >= 0
 c4:  x3 + x4 + x5 + x10 - x28 >= 0
 c5:  x4 + x5 + x6 + x11 - x29 >= 0
 c6:  x5 + x6 + x12 - x30 >= 0
 c7:  x1 + x7 + x8 + x13 - x31 >= 0
 c8:  x2 + x7 + x8 + x9 + x14 - x32 >= 0
 c9:  x3 + x8 + x9 + x10 + x15 - x33 >= 0
 c10: x4 + x9 + x10 + x11 + x16 - x34 >= 0
 c11: x5 + x10 + x11 + x12 + x17 - x35 >= 0
 c12: x6 + x11 + x12 + x18 - x36 >= 0
 c13: x7 + x13 + x14 + x19 - x37 >= 0
 c14: x8 + x13 + x14 + x15 + x20 - x38 >= 0
 c15: x9 + x14 + x15 + x16 + x21 - x39 >= 0
 c16: x10 + x15 + x16 + x17 + x22 - x40 >= 0
 c17: x11 + x16 + x17 + x18 + x23 - x41 >= 0
 c18: x12 + x